# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

In [1]:
import os
import langsmith
import getpass

# Set up LangSmith for tracking and evaluation
os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API Key:")
os.environ["LANGSMITH_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Advanced_Retrieval_Session09"

# Initialize LangSmith client
from langsmith import Client
langsmith_client = Client()

print("LangSmith setup complete!")


Enter your LangSmith API Key: ········


LangSmith setup complete!


## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key: ········


In [3]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

Cohere API Key: ········


## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [92]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/Projects_with_Domains.csv",
    metadata_columns=[
      "Project Title",
      "Project Domain",
      "Secondary Domain",
      "Description",
      "Judge Comments",
      "Score",
      "Project Name",
      "Judge Score"
    ]
)

synthetic_usecase_data = loader.load()

for doc in synthetic_usecase_data:
    doc.page_content = doc.metadata["Description"]

Let's look at an example document to see if everything worked as expected!

In [93]:
synthetic_usecase_data[0]

Document(metadata={'source': './data/Projects_with_Domains.csv', 'row': 0, 'Project Title': 'InsightAI 1', 'Project Domain': 'Security', 'Secondary Domain': 'Finance / FinTech', 'Description': 'A low-latency inference system for multimodal agents in autonomous systems.', 'Judge Comments': 'Technically ambitious and well-executed.', 'Score': '85', 'Project Name': 'Project Aurora', 'Judge Score': '9.5'}, page_content='A low-latency inference system for multimodal agents in autonomous systems.')

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [94]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [95]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [96]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [97]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [98]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [99]:
naive_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "naive", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned in multiple entries.'

In [100]:
naive_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "naive", "question_type": "security_query"}}
)["response"].content

'Yes, there are usecases related to security in the provided data. Specifically, one project titled "Pathfinder 24" is described as "An AI-powered platform optimizing logistics routes for sustainability," with a secondary domain of Security.'

In [101]:
naive_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "naive", "question_type": "judge_feedback"}}
)["response"].content

'The judges\' comments on the fintech projects were generally positive. For example, one project described as a "federated learning toolkit improving privacy in healthcare applications" was praised as "a clever solution with measurable environmental benefit." Another project received the comment "Solid work with impressive real-world impact," and yet another was noted as "Technically ambitious and well-executed." Additionally, a project involving "an AI model compression suite enabling on-device reasoning for IoT sensors" was acknowledged for its "strong quantitative results" with a suggestion to add qualitative analysis. Overall, judges highlighted the projects\' strong technical quality, potential real-world impact, and innovative approach.'

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [102]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [103]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [104]:
bm25_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "bm25", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain is not explicitly stated, but from the sample projects listed, the domains include "Productivity Assistants," "E-commerce / Marketplaces," "Healthcare / MedTech," and "Finance / FinTech." Since the sample size is small and no domain appears repeatedly, I cannot determine the most common project domain with certainty from this data alone. If you have access to the full dataset, analyzing the frequency of each domain would provide an accurate answer.'

In [105]:
bm25_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "bm25", "question_type": "security_query"}}
)["response"].content

'Yes, there was a use case related to security. The project "SecureNest" falls under the domains of E-commerce / Marketplaces and Legal / Compliance, and it involves a document summarization and retrieval system for enterprise knowledge bases, which can be related to security and compliance needs.'

In [106]:
bm25_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "bm25", "question_type": "judge_feedback"}}
)["response"].content

'The judges described the fintech project, "SynthMind," as having a strong conceptual foundation, but noted that its results require more benchmarking.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer
 1. When exact term matching is needed e.g.  supervised, unsupervise machine learning have specific meaning, that may be missed using semantic retrieval
 2. Finding frequency count of a term. When this is required semantic retrieval will distort frequency count.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [107]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [108]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [109]:
contextual_compression_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "compression", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is explicitly mentioned in at least one of the example entries. However, since the data sample is limited and only shows a few projects, I cannot definitively determine the most common domain overall. If you have access to the full dataset, analyzing the frequency of each project domain would give a precise answer.'

In [110]:
contextual_compression_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "compression", "question_type": "security_query"}}
)["response"].content

'Based on the provided context, there are no specific usecases related to security mentioned. The projects focus on federated learning to improve privacy in healthcare applications, but security as a distinct usecase is not explicitly stated.'

In [111]:
contextual_compression_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "compression", "question_type": "judge_feedback"}}
)["response"].content

'Judges had positive comments about the fintech projects. Specifically, for the Pathfinder 27 project in the fintech domain, judges praised it for "excellent code quality and use of open-source libraries," with a high judge score of 9.8.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [112]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [113]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [114]:
multi_query_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "multi_query", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "E‑commerce / Marketplaces," which is listed multiple times in the dataset.'

In [115]:
multi_query_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "multi_query", "question_type": "security_query"}}
)["response"].content

'Yes, there are use cases related to security. Specifically, there is a project titled "Pathfinder 24" in the Healthcare / MedTech domain with a secondary domain of Security. Its description is "An AI-powered platform optimizing logistics routes for sustainability," which suggests a focus on security aspects related to logistics and transportation.'

In [116]:
multi_query_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "multi_query", "question_type": "judge_feedback"}}
)["response"].content

'The judges had varied comments on the fintech projects. For the project "GreenPulse," they described it as "Technically ambitious and well-executed" with a high judge score of 8.9. Similarly, "DataWeave" was praised for its "Excellent code quality and use of open-source libraries," earning a judge score of 9.8. The project "SynthMind" was recognized for being "Conceptually strong but results need more benchmarking," receiving a judge score of 9.6, and "LatticeFlow" was noted as "Well-structured and scalable; good potential for commercialization," with a judge score of 7.7. Overall, the judges viewed these fintech-related projects positively, highlighting their technical ambition, quality, and potential impact.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer

 1. Multi-query allows slightly different retrieval around similar semantic meaning. Different phrasing of same concept wiii be retrieved.
 2. Recall is the percentage of possible relevant answers being recalled.
 3. More retrieval leads to more likelihood of finding the true/relevant answers.


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [117]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [118]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [119]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [120]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [121]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [122]:
parent_document_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "parent_document", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned more than once among the projects listed.'

In [123]:
parent_document_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "parent_document", "question_type": "security_query"}}
)["response"].content

'Based on the provided context, there are no specific use cases related to security explicitly mentioned. The projects listed primarily focus on federated learning and privacy improvements in healthcare applications, but none are explicitly described as security use cases.'

In [124]:
parent_document_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "parent_document", "question_type": "judge_feedback"}}
)["response"].content

'The judges had positive comments about the fintech projects, often highlighting their technical quality and real-world impact. For example, one judge described a project as "Solid work with impressive real-world impact," while another called a project "Technically ambitious and well-executed." Overall, the judge comments suggest a recognition of the projects\' innovation, robustness, and potential influence.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [125]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [126]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [127]:
ensemble_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "ensemble", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "Healthcare / MedTech," as it is mentioned multiple times in the sample. However, since only a subset of data is shown, I cannot confirm with absolute certainty that it is the most common overall. But among the displayed entries, "Healthcare / MedTech" is the most frequently occurring domain.'

In [128]:
ensemble_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "ensemble", "question_type": "security_query"}}
)["response"].content

'Yes, there is at least one usecase related to security. The project titled "SecureNest" is described as a "document summarization and retrieval system for enterprise knowledge bases," which falls under the domain of Legal / Compliance and involves security aspects.'

In [129]:
ensemble_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "ensemble", "question_type": "judge_feedback"}}
)["response"].content

'Judges had various opinions on the fintech projects. For example, they described the project "DocuCheck" as conceptually strong, although noting that its results need more benchmarking. Overall, judges recognized the innovative ideas and technical maturity of the projects in this domain, but also pointed out areas for improvement such as the need for stronger evaluation metrics or additional benchmarking.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [130]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [131]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [132]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [133]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [134]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [135]:
semantic_retrieval_chain.invoke(
    {"question" : "What is the most common project domain?"},
    config={"metadata": {"retriever": "semantic", "question_type": "domain_analysis"}}
)["response"].content

'Based on the provided data, the most common project domain appears to be "Legal / Compliance," as it is listed more than once among the projects. However, it is also important to note that "Developer Tools / DevEx" and "Customer Support / Helpdesk" are mentioned multiple times. Without a full count of all entries, I cannot confirm definitively, but from the sample, "Legal / Compliance" seems to be the most frequently occurring domain.'

In [136]:
semantic_retrieval_chain.invoke(
    {"question" : "Were there any usecases about security?"},
    config={"metadata": {"retriever": "semantic", "question_type": "security_query"}}
)["response"].content

'Yes, there are use cases related to security in the provided data. Specifically, the project "MediMind 17" titled "BioForge" is a medical imaging solution that improves early diagnosis through vision transformers and is categorized under the Security domain. Additionally, "InsightAI 1" titled "Project Aurora" is a low-latency inference system for multimodal agents in autonomous systems, also under the Security domain.'

In [137]:
semantic_retrieval_chain.invoke(
    {"question" : "What did judges have to say about the fintech projects?"},
    config={"metadata": {"retriever": "semantic", "question_type": "judge_feedback"}}
)["response"].content

'The judges had various comments about the fintech projects. For example, they described the "TrendLens 19" project as "Technically ambitious and well-executed," and the "WealthifyAI 16" project as having a "Comprehensive and technically mature approach." Overall, the judges\' remarks highlight qualities such as ambition, technical soundness, and maturity in these projects.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer

 1. For FAQs specifically, there is predictable structure in (Q: ... A: ...)
 2. Clear boundaries between questions, answers
 3. Semantic similarity is misleading (all Q&As are "related" to the topic)

In short clear partitioning , regex methods would be idea. In langchain this is implemented wtih RecursiveCharacterTextSplitter. It's designed for exactly this scenario - structured, repetitive content where semantic boundaries don't align with logical content boundaries.


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

#### For solution to activity #1 see file compare2.ipynb